In [1]:
from flask import Flask, request, jsonify, send_file
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration
from gtts import gTTS
import os

app = Flask(__name__)

# Load BLIP Model and Processor
blip_processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
blip_model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

# Generate Caption
def generate_caption(image):
    inputs = blip_processor(image, return_tensors="pt")
    outputs = blip_model.generate(**inputs)
    caption = blip_processor.decode(outputs[0], skip_special_tokens=True)
    return caption

# Generate Audio
def generate_audio(caption, output_path):
    tts = gTTS(caption, lang="en")
    tts.save(output_path)
    return output_path

@app.route("/caption", methods=["POST"])
def caption_image():
    if "image" not in request.files:
        return jsonify({"error": "No image uploaded"}), 400

    image_file = request.files["image"]
    image = Image.open(image_file).convert("RGB")

    # Generate caption
    caption = generate_caption(image)

    # Generate audio
    audio_path = "output.mp3"
    generate_audio(caption, audio_path)

    # Send caption and audio
    response = {
        "caption": caption
    }
    return send_file(audio_path, mimetype="audio/mpeg", as_attachment=True, attachment_filename="caption_audio.mp3")

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)


ModuleNotFoundError: No module named 'gtts'